# Audio + Flow Tensor Motion Anomaly Training

音声モデルは従来通りIsolationForestで学習し、動きモデルは `MOTION_FEATURE_USAGE` で選択したflow系特徴の3x3時間tensorで学習します。音声・動き・同期スコアを統合したartifactを保存します。


## 処理の流れ

1. 学習対象sampleを選び、音声特徴と0.1秒binのflow tensorを共通pipelineで抽出します。
2. 音声IsolationForestとmotion tensor正常モデルを学習します。
3. 音声・動き・同期スコアを統合し、推論Notebookで使うartifactを保存します。


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        fallback_iterable = [] if iterable is None else iterable
        return fallback_iterable


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'forklift_features').exists():
            return candidate
    raise FileNotFoundError(f'Could not find src/forklift_features from {start}')


PROJECT_ROOT = _find_import_root()
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import flow_tensor, pipeline as feature_pipeline, scoring
from forklift_features.paths import find_project_root

flow_tensor = importlib.reload(flow_tensor)
feature_pipeline = importlib.reload(feature_pipeline)
scoring = importlib.reload(scoring)

pd.set_option('display.max_columns', 160)


In [ ]:
PROJECT_ROOT = find_project_root()
MOVIE_PREPROCESS_DIR = PROJECT_ROOT / 'data' / 'movie_preprocess'
TRAIN_DATA_DIR = MOVIE_PREPROCESS_DIR / 'normal'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'anomaly_feature_poc'
FEATURE_CACHE_DIR = OUTPUT_DIR / 'sample_feature_cache'
MODEL_PATH = OUTPUT_DIR / 'isolation_forest_feature_poc.joblib'
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'train_sample_list.csv'
TRAIN_TARGET_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'target_sample_list.csv'
TRAIN_TARGET_SAMPLE_LIST_LEGACY_PATH = PROJECT_ROOT / 'data' / 'target_sample_list'
if not TRAIN_TARGET_SAMPLE_LIST_PATH.exists() and TRAIN_TARGET_SAMPLE_LIST_LEGACY_PATH.exists():
    TRAIN_TARGET_SAMPLE_LIST_PATH = TRAIN_TARGET_SAMPLE_LIST_LEGACY_PATH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_SR = 16000
N_MELS = 16
WINDOW_SEC = 0.2

FLOW_SAMPLE_SEC = 0.10
FLOW_GRID = (3, 3)
USE_FRONT = True
USE_REAR = True
FRAME_RESIZE_WIDTH = 480
FLOW_ANALYSIS_SCALE = 0.75
FLOW_RELIABLE_ERROR_THRESHOLD_PX = 1.0
FLOW_TENSOR_WINDOW_SEC = 1.0
FLOW_TENSOR_HOP_SEC = 0.2
FLOW_TENSOR_SCORE_AGGREGATION = 'max'
MOTION_FEATURE_USAGE = {
    'flow_x': False,
    'flow_y': False,
    't_flow_x': False,
    't_flow_y': False,
    'flow_x_broad_vib_score': False,
    'flow_y_broad_vib_score': False,
    't_flow_x_broad_vib_score': True,
    't_flow_y_broad_vib_score': True,
}
MOTION_FEATURE_NAMES = [name for name, enabled in MOTION_FEATURE_USAGE.items() if enabled]
if not MOTION_FEATURE_NAMES:
    raise ValueError('At least one motion feature must be enabled in MOTION_FEATURE_USAGE.')
FLOW_TENSOR_BROAD_VIB_SCORE_CONFIG = {
    'high_ratio_fraction': 0.5,
    'lower_percentile': 0.0,
    'upper_percentile': 95.0,
    'min_visible': 1e-6,
    'broad_vib_score_weights': {'A': 0.25, 'B': 0.75, 'C': 0.0},
    'broad_vib_low_intensity_percentile': 20.0,
}
FLOW_CACHE_VERSION = 'sample_flow_cache_v5'
FLOW_CACHE_METHOD = 'farneback_grid_frame_bin_apportioned_0p1s'

SCORE_CALIBRATION_QUANTILES = (0.50, 0.995)
MODEL_INPUT_DTYPE = np.float32
RANDOM_STATE = 42
MAX_TRAIN_VIDEOS = 50
TRAIN_SELECTION_MODE = 'sample_list' if TRAIN_TARGET_SAMPLE_LIST_PATH.exists() else 'random'  # 'sample_list', 'manual', or 'random'
TRAIN_SPECS: list[dict[str, Any]] = []
RANDOM_TRAIN_COUNT = MAX_TRAIN_VIDEOS
RANDOM_TRAIN_ENVIRONMENT_RATIOS = {'indoor': 0.8, 'outdoor': 0.2}

SYNC_SCORE_CONFIG = {
    'audio_score_column': 'audio_anomaly_score',
    'motion_score_column': 'motion_anomaly_score',
    'max_lag_windows': 2,
}
FINAL_SCORE_WEIGHTS = {
    'audio_anomaly_score': 0.45,
    'motion_anomaly_score': 0.35,
    'sync_score': 0.20,
}
PLOT_SCORE_COLUMNS = ['anomaly_score', 'audio_anomaly_score', 'motion_anomaly_score', 'sync_score']

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'TRAIN_DATA_DIR: {TRAIN_DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'TRAIN_SELECTION_MODE: {TRAIN_SELECTION_MODE}')
print(f'TRAIN_TARGET_SAMPLE_LIST_PATH: {TRAIN_TARGET_SAMPLE_LIST_PATH}')


In [ ]:
def list_movie_preprocess_samples(root: Path = TRAIN_DATA_DIR) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for front_path in sorted(root.rglob('*_front.mp4')):
        sample_id = front_path.name[:-len('_front.mp4')]
        rear_path = front_path.with_name(f'{sample_id}_rear.mp4')
        audio_path = front_path.with_name(f'{sample_id}_audio.wav')
        if not rear_path.exists():
            continue
        try:
            rel_parts = front_path.relative_to(MOVIE_PREPROCESS_DIR).parts
            category = rel_parts[0] if len(rel_parts) >= 1 else 'normal'
            environment = rel_parts[1] if len(rel_parts) >= 2 else front_path.parent.name
        except ValueError:
            category = 'normal'
            environment = front_path.parent.name
        rows.append({
            'sample_id': sample_id,
            'category': category,
            'environment': environment,
            'front_path': front_path,
            'rear_path': rear_path,
            'audio_path': audio_path if audio_path.exists() else None,
            'plot_label': sample_id,
        })
    return pd.DataFrame(rows)


def select_manual_samples(samples_df: pd.DataFrame, specs: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for spec in specs:
        part = samples_df[samples_df['sample_id'].astype(str).eq(str(spec['sample_id']))]
        if 'environment' in spec:
            part = part[part['environment'].astype(str).eq(str(spec['environment']))]
        if 'category' in spec:
            part = part[part['category'].astype(str).eq(str(spec['category']))]
        if len(part) != 1:
            raise ValueError(f'sample spec must match exactly one row: {spec}, matches={len(part)}')
        rows.append(part.iloc[0])
    return pd.DataFrame(rows).reset_index(drop=True)


def rows_from_target_sample_specs(samples_df: pd.DataFrame, specs_df: pd.DataFrame) -> pd.DataFrame:
    if 'sample_id' not in specs_df.columns:
        raise ValueError('target_sample_list must contain a sample_id column')
    rows = []
    for _, spec in specs_df.iterrows():
        sample_id = str(spec.get('sample_id')).strip()
        if not sample_id or sample_id.lower() == 'nan':
            continue
        if {'front_path', 'rear_path'}.issubset(specs_df.columns) and pd.notna(spec.get('front_path')) and pd.notna(spec.get('rear_path')):
            front_path = Path(str(spec.get('front_path')))
            rear_path = Path(str(spec.get('rear_path')))
            audio_value = spec.get('audio_path') if 'audio_path' in specs_df.columns else None
            audio_path = Path(str(audio_value)) if pd.notna(audio_value) else front_path.with_name(f'{sample_id}_audio.wav')
            rows.append({
                'sample_id': sample_id,
                'category': str(spec.get('category', 'normal')),
                'environment': str(spec.get('environment', 'unknown')),
                'front_path': front_path,
                'rear_path': rear_path,
                'audio_path': audio_path if audio_path.exists() else None,
                'plot_label': str(spec.get('plot_label', sample_id)),
            })
            continue

        part = samples_df[samples_df['sample_id'].astype(str).eq(sample_id)].copy()
        if 'category' in specs_df.columns and pd.notna(spec.get('category')):
            part = part[part['category'].astype(str).eq(str(spec.get('category')))]
        if 'environment' in specs_df.columns and pd.notna(spec.get('environment')):
            part = part[part['environment'].astype(str).eq(str(spec.get('environment')))]
        if len(part) != 1:
            raise ValueError(f'target_sample_list row must match exactly one training row: {dict(spec)}, matches={len(part)}')
        row = part.iloc[0].copy()
        if 'plot_label' in specs_df.columns and pd.notna(spec.get('plot_label')):
            row['plot_label'] = spec.get('plot_label')
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def load_target_sample_list(path: Path, samples_df: pd.DataFrame) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'target_sample_list was not found: {path}')
    specs_df = pd.read_csv(path)
    return rows_from_target_sample_specs(samples_df, specs_df)


def select_random_samples(samples_df: pd.DataFrame) -> pd.DataFrame:
    count = len(samples_df) if RANDOM_TRAIN_COUNT is None or int(RANDOM_TRAIN_COUNT) < 0 else min(int(RANDOM_TRAIN_COUNT), len(samples_df))
    if count <= 0:
        return samples_df.head(0).copy()
    selected_parts = []
    remaining = samples_df.copy()
    for environment, ratio in RANDOM_TRAIN_ENVIRONMENT_RATIOS.items():
        env_part = remaining[remaining['environment'].astype(str).eq(str(environment))]
        n = min(int(round(count * float(ratio))), len(env_part))
        if n > 0:
            part = env_part.sample(n=n, random_state=RANDOM_STATE)
            selected_parts.append(part)
            remaining = remaining.drop(index=part.index)
    selected = pd.concat(selected_parts, ignore_index=False) if selected_parts else samples_df.head(0)
    if len(selected) < count and len(remaining):
        selected = pd.concat([selected, remaining.sample(n=min(count - len(selected), len(remaining)), random_state=RANDOM_STATE)])
    return selected.sort_values(['environment', 'sample_id']).reset_index(drop=True)


all_samples_df = list_movie_preprocess_samples()
if TRAIN_SELECTION_MODE == 'sample_list':
    train_samples = load_target_sample_list(TRAIN_TARGET_SAMPLE_LIST_PATH, all_samples_df)
    selection_source = f'target_sample_list: {TRAIN_TARGET_SAMPLE_LIST_PATH}'
elif TRAIN_SELECTION_MODE == 'manual':
    if not TRAIN_SPECS:
        raise ValueError("TRAIN_SPECS must not be empty when TRAIN_SELECTION_MODE='manual'")
    train_samples = select_manual_samples(all_samples_df, TRAIN_SPECS)
    selection_source = 'manual'
elif TRAIN_SELECTION_MODE == 'random':
    train_samples = select_random_samples(all_samples_df)
    selection_source = 'random'
else:
    raise ValueError("TRAIN_SELECTION_MODE must be 'sample_list', 'manual', or 'random'")

if train_samples.empty:
    raise ValueError('No training samples were selected')

TRAIN_SAMPLE_LIST_PATH.parent.mkdir(parents=True, exist_ok=True)
train_samples[['sample_id', 'category', 'environment', 'front_path', 'rear_path', 'audio_path']].to_csv(TRAIN_SAMPLE_LIST_PATH, index=False)
display(train_samples[['sample_id', 'category', 'environment', 'front_path', 'rear_path', 'audio_path']])
print(f'selection source: {selection_source}')
print(f'saved train sample list: {TRAIN_SAMPLE_LIST_PATH}')


In [ ]:
# ------------------------------------------------------------
# セル概要: 共通特徴抽出パイプライン設定
# ------------------------------------------------------------
# - flow cache設定、motion tensor抽出設定、音声特徴抽出設定をNotebook固有の定数から作ります。
# - 実際の抽出・cache利用・音声/動きスコア結合は src/forklift_features/pipeline.py に集約しています。
# ------------------------------------------------------------

FLOW_CACHE_SETTINGS = feature_pipeline.build_flow_cache_settings(
    cache_version=FLOW_CACHE_VERSION,
    use_front=USE_FRONT,
    use_rear=USE_REAR,
    flow_sample_sec=FLOW_SAMPLE_SEC,
    frame_resize_width=FRAME_RESIZE_WIDTH,
    flow_analysis_scale=FLOW_ANALYSIS_SCALE,
    flow_grid=FLOW_GRID,
    flow_reliable_error_threshold_px=FLOW_RELIABLE_ERROR_THRESHOLD_PX,
    flow_method=FLOW_CACHE_METHOD,
)
MOTION_PIPELINE_KWARGS = {
    'flow_cache_settings': FLOW_CACHE_SETTINGS,
    'cache_dir': FEATURE_CACHE_DIR,
    'use_front': USE_FRONT,
    'use_rear': USE_REAR,
    'flow_sample_sec': FLOW_SAMPLE_SEC,
    'frame_resize_width': FRAME_RESIZE_WIDTH,
    'flow_analysis_scale': FLOW_ANALYSIS_SCALE,
    'flow_grid': FLOW_GRID,
    'flow_reliable_error_threshold_px': FLOW_RELIABLE_ERROR_THRESHOLD_PX,
    'tensor_window_sec': FLOW_TENSOR_WINDOW_SEC,
    'tensor_hop_sec': FLOW_TENSOR_HOP_SEC,
    'motion_feature_names': MOTION_FEATURE_NAMES,
    'broad_vib_score_config': FLOW_TENSOR_BROAD_VIB_SCORE_CONFIG,
    'label_default': 'normal',
    'enable_cache': True,
}
AUDIO_PIPELINE_KWARGS = {
    'audio_sr': AUDIO_SR,
    'window_sec': WINDOW_SEC,
    'hop_sec': FLOW_TENSOR_HOP_SEC,
    'n_mels': N_MELS,
    'label_default': 'normal',
}


In [ ]:
X_parts: list[np.ndarray] = []
motion_meta_parts: list[pd.DataFrame] = []
audio_feature_parts: list[pd.DataFrame] = []
cache_rows: list[dict[str, Any]] = []

for record in tqdm(train_samples.to_dict('records'), total=len(train_samples), desc='extract train features'):
    sample = pd.Series(record)
    try:
        X_sample, meta_sample, _raw_flow_df, cache_info = feature_pipeline.build_motion_windows(sample, **MOTION_PIPELINE_KWARGS)
        cache_rows.append(cache_info)
        if len(X_sample):
            X_parts.append(X_sample)
            motion_meta_parts.append(meta_sample)
        audio_df = feature_pipeline.build_audio_features(sample, **AUDIO_PIPELINE_KWARGS)
        if len(audio_df):
            audio_feature_parts.append(audio_df)
    except Exception as exc:
        cache_rows.append({'sample_id': str(sample.get('sample_id', 'unknown')), 'cache_status': 'error', 'error': repr(exc)})

flow_cache_manifest_df = pd.DataFrame(cache_rows)
train_motion_meta_df = pd.concat(motion_meta_parts, ignore_index=True) if motion_meta_parts else pd.DataFrame()
train_audio_features_df = pd.concat(audio_feature_parts, ignore_index=True) if audio_feature_parts else pd.DataFrame()
X_train_motion = np.concatenate(X_parts, axis=0).astype(MODEL_INPUT_DTYPE, copy=False) if X_parts else np.zeros((0, 1, *FLOW_GRID, len(MOTION_FEATURE_NAMES)), dtype=MODEL_INPUT_DTYPE)

flow_cache_manifest_path = OUTPUT_DIR / 'flow_cache_manifest.csv'
flow_cache_manifest_df.to_csv(flow_cache_manifest_path, index=False)
if len(flow_cache_manifest_df):
    display(flow_cache_manifest_df['cache_status'].value_counts().rename_axis('cache_status').reset_index(name='count'))
print(f'saved: {flow_cache_manifest_path}')
print(f'X_train_motion shape: {X_train_motion.shape}')
print(f'audio feature shape: {train_audio_features_df.shape}')


In [ ]:
audio_feature_cols = [c for c in train_audio_features_df.columns if c.startswith('audio_') and c != 'audio_window_index']
if not audio_feature_cols:
    raise ValueError('No audio feature columns were produced. Check *_audio.wav files.')

audio_preprocess_pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
X_audio = train_audio_features_df[audio_feature_cols].replace([np.inf, -np.inf], np.nan)
X_audio_model = audio_preprocess_pipeline.fit_transform(X_audio)
audio_model = IsolationForest(n_estimators=200, contamination='auto', random_state=RANDOM_STATE, n_jobs=-1)
audio_model.fit(X_audio_model)
audio_raw_scores = scoring.isolation_forest_raw_scores(audio_model, X_audio_model)
audio_calibration = scoring.fit_score_calibration(audio_raw_scores, SCORE_CALIBRATION_QUANTILES)
audio_scores = scoring.apply_score_calibration(audio_raw_scores, audio_calibration)

train_audio_scores_df = train_audio_features_df.copy()
train_audio_scores_df['audio_anomaly_score_raw'] = audio_raw_scores
train_audio_scores_df['audio_anomaly_score'] = audio_scores

flow_tensor_model, motion_raw_scores, motion_scores = flow_tensor.fit_normal_flow_tensor_model(
    X_train_motion,
    feature_names=MOTION_FEATURE_NAMES,
    calibration_quantiles=SCORE_CALIBRATION_QUANTILES,
)
train_motion_window_scores_df = train_motion_meta_df.copy()
train_motion_window_scores_df['motion_anomaly_score_raw'] = motion_raw_scores
train_motion_window_scores_df['motion_anomaly_score'] = motion_scores
motion_explanation_df = flow_tensor.explain_flow_tensor_windows(
    X_train_motion,
    flow_tensor_model,
    train_motion_meta_df,
)
if len(motion_explanation_df):
    for col in motion_explanation_df.columns:
        train_motion_window_scores_df[col] = motion_explanation_df[col].to_numpy()
train_motion_window_scores_df['anomaly_score'] = motion_scores
train_motion_window_scores_df['anomaly_score_smooth'] = train_motion_window_scores_df.groupby(['sample_id', 'camera'])['anomaly_score'].transform(lambda s: s.rolling(window=5, center=True, min_periods=1).mean())
train_motion_scores_df = flow_tensor.aggregate_camera_scores(
    train_motion_window_scores_df,
    method=FLOW_TENSOR_SCORE_AGGREGATION,
    score_col='motion_anomaly_score',
    raw_score_col='motion_anomaly_score_raw',
)

train_scored_df = feature_pipeline.combine_audio_motion_scores(train_audio_scores_df, train_motion_scores_df, hop_sec=FLOW_TENSOR_HOP_SEC)
train_scored_df = scoring.add_composite_scores(
    train_scored_df,
    sync_score_config=SYNC_SCORE_CONFIG,
    final_score_weights=FINAL_SCORE_WEIGHTS,
)

video_scores_df = (
    train_scored_df
    .groupby(['video_id', 'target_category', 'target_environment', 'target_label'], as_index=False)
    .agg(
        max_anomaly_score=('anomaly_score', 'max'),
        top5_mean_anomaly_score=('anomaly_score', scoring.topk_mean),
        p95_anomaly_score=('anomaly_score', lambda s: float(s.quantile(0.95))),
        max_audio_anomaly_score=('audio_anomaly_score', 'max'),
        max_motion_anomaly_score=('motion_anomaly_score', 'max'),
        max_sync_score=('sync_score', 'max'),
        n_windows=('anomaly_score', 'size'),
    )
    .sort_values('max_anomaly_score', ascending=False)
    .reset_index(drop=True)
)

feature_catalog_df = pd.DataFrame([
    {'feature_name': col, 'feature_group': 'audio'} for col in audio_feature_cols
] + [
    {'feature_name': name, 'feature_group': 'motion_flow_tensor'} for name in MOTION_FEATURE_NAMES
])

artifact = {
    'model_version': 'audio_flow_tensor_sync_v2',
    'model_type': 'audio_flow_tensor_sync',
    'flow_tensor_model': flow_tensor_model,
    'score_models': {
        'audio': {
            'model_name': 'audio',
            'model': audio_model,
            'preprocess_pipeline': audio_preprocess_pipeline,
            'feature_names': audio_feature_cols,
            'score_calibration': audio_calibration,
            'score_column': 'audio_anomaly_score',
            'raw_score_column': 'audio_anomaly_score_raw',
            'model_input_shape': tuple(X_audio_model.shape),
        },
        'motion': {
            'model_name': 'motion_flow_tensor',
            'model_type': 'normal_flow_tensor_zscore',
            'flow_tensor_model': flow_tensor_model,
            'score_column': 'motion_anomaly_score',
            'raw_score_column': 'motion_anomaly_score_raw',
            'tensor_shape': flow_tensor_model.get('tensor_shape'),
            'channels': list(MOTION_FEATURE_NAMES),
        },
    },
    'score_model_configs': {
        'audio': {'enabled': True, 'score_column': 'audio_anomaly_score', 'raw_score_column': 'audio_anomaly_score_raw'},
        'motion': {'enabled': True, 'score_column': 'motion_anomaly_score', 'raw_score_column': 'motion_anomaly_score_raw', 'model_type': 'normal_flow_tensor_zscore'},
    },
    'sync_score_config': SYNC_SCORE_CONFIG,
    'final_score_weights': FINAL_SCORE_WEIGHTS,
    'plot_score_columns': PLOT_SCORE_COLUMNS,
    'feature_catalog': feature_catalog_df,
    'settings': {
        'audio_sr': AUDIO_SR,
        'n_mels': N_MELS,
        'window_sec': WINDOW_SEC,
        'flow_sample_sec': FLOW_SAMPLE_SEC,
        'flow_grid': FLOW_GRID,
        'use_front': USE_FRONT,
        'use_rear': USE_REAR,
        'frame_resize_width': FRAME_RESIZE_WIDTH,
        'flow_analysis_scale': FLOW_ANALYSIS_SCALE,
        'flow_reliable_error_threshold_px': FLOW_RELIABLE_ERROR_THRESHOLD_PX,
        'flow_tensor_window_sec': FLOW_TENSOR_WINDOW_SEC,
        'flow_tensor_hop_sec': FLOW_TENSOR_HOP_SEC,
        'flow_tensor_score_aggregation': FLOW_TENSOR_SCORE_AGGREGATION,
        'motion_feature_usage': MOTION_FEATURE_USAGE,
        'motion_feature_names': list(MOTION_FEATURE_NAMES),
        'flow_tensor_broad_vib_score_config': FLOW_TENSOR_BROAD_VIB_SCORE_CONFIG,
        'score_calibration_quantiles': SCORE_CALIBRATION_QUANTILES,
        'model_input_dtype': str(np.dtype(MODEL_INPUT_DTYPE)),
        'flow_cache_version': FLOW_CACHE_VERSION,
        'flow_cache_settings': FLOW_CACHE_SETTINGS,
    },
    'train_sample_ids': train_samples['sample_id'].astype(str).tolist(),
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)
train_audio_features_df.to_csv(OUTPUT_DIR / 'features.csv', index=False)
train_scored_df.to_csv(OUTPUT_DIR / 'anomaly_scores.csv', index=False)
video_scores_df.to_csv(OUTPUT_DIR / 'video_scores.csv', index=False)
feature_catalog_df.to_csv(OUTPUT_DIR / 'feature_catalog.csv', index=False)
feature_catalog_df[feature_catalog_df['feature_group'].eq('audio')].to_csv(OUTPUT_DIR / 'feature_catalog_audio.csv', index=False)
feature_catalog_df[feature_catalog_df['feature_group'].ne('audio')].to_csv(OUTPUT_DIR / 'feature_catalog_motion.csv', index=False)
train_motion_window_scores_df.to_csv(OUTPUT_DIR / 'flow_tensor_train_window_scores.csv', index=False)
train_motion_scores_df.to_csv(OUTPUT_DIR / 'flow_tensor_train_sample_scores.csv', index=False)

print(f'saved model: {MODEL_PATH}')
print(f'tensor shape: {flow_tensor_model["tensor_shape"]}')
print(f'motion features: {MOTION_FEATURE_NAMES}')
motion_attribution_display_cols = [
    'motion_top_camera',
    'motion_top_grid_channel',
    'motion_top_grid_label',
    'motion_top_grid_contribution',
    'motion_flow_x_contribution',
    'motion_flow_y_contribution',
    'motion_t_flow_x_contribution',
    'motion_t_flow_y_contribution',
    'motion_flow_x_broad_vib_score_contribution',
    'motion_flow_y_broad_vib_score_contribution',
    'motion_t_flow_x_broad_vib_score_contribution',
    'motion_t_flow_y_broad_vib_score_contribution',
]
train_display_cols = [c for c in [
    'video_id', 'target_category', 'target_environment', 'time', 'window_start_sec',
    'anomaly_score', 'audio_anomaly_score', 'motion_anomaly_score', 'sync_score',
    *motion_attribution_display_cols,
] if c in train_scored_df.columns]
display(train_scored_df.sort_values('anomaly_score', ascending=False)[train_display_cols].head(20))
display(video_scores_df.head(20))
